# Deep Learning Models for Oil Recovery Factor Prediction
## Comparative Study: MLP · LSTM · CNN-1D · Transformer

**Dataset:** Proxy5 — polymer flood reservoir simulation  
**Target:** `Oil_recovery_factor (%)`  
**Features:** 14 reservoir / fluid / operational parameters  
**Framework:** Keras (TensorFlow back-end)

---
## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import keras
from keras import layers, Model, Input
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.utils import plot_model

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {keras.__version__}')
print(f'GPU available: {bool(tf.config.list_physical_devices("GPU"))}')

# ── Shared hyper-parameters ──────────────────────────────────────────────────
BATCH_SIZE = 64
EPOCHS     = 200
LR         = 1e-3
PATIENCE   = 20
TEST_SIZE  = 0.15
VAL_SIZE   = 0.15

COLOR_MAP = {'MLP': '#2196F3', 'CNN-1D': '#FF9800',
             'LSTM': '#4CAF50', 'Transformer': '#9C27B0'}

---
## 2. Data Loading & Exploration

In [ ]:
DATA_PATH = 'Proxy5.csv'
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().T.style.background_gradient(cmap='YlGnBu', axis=1)

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
TARGET   = 'Oil_recovery_factor (%)'
FEATURES = [c for c in df.columns if c != TARGET]
print(f'Features ({len(FEATURES)}):', FEATURES)

### 2.1 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram
axes[0].hist(df[TARGET], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Histogram', fontsize=12)
axes[0].set_xlabel(TARGET)
axes[0].set_ylabel('Count')

# Box plot
axes[1].boxplot(df[TARGET], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='navy', linewidth=2))
axes[1].set_title('Box Plot', fontsize=12)
axes[1].set_ylabel(TARGET)
axes[1].set_xticklabels(['Oil Recovery Factor'])

# Cumulative distribution
sorted_vals = np.sort(df[TARGET])
cdf = np.arange(1, len(sorted_vals)+1) / len(sorted_vals)
axes[2].plot(sorted_vals, cdf, color='steelblue', lw=2)
axes[2].set_title('Cumulative Distribution (CDF)', fontsize=12)
axes[2].set_xlabel(TARGET)
axes[2].set_ylabel('Cumulative Probability')
axes[2].grid(alpha=0.3)

plt.suptitle('Target Variable Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Feature Distributions

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()
for i, feat in enumerate(FEATURES):
    axes[i].hist(df[feat], bins=40, color='#5C85D6', edgecolor='white', alpha=0.85)
    axes[i].set_title(feat, fontsize=8, fontweight='bold')
    axes[i].tick_params(labelsize=7)
# Hide spare axes
for j in range(len(FEATURES), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Correlation Heatmap

In [ ]:
plt.figure(figsize=(15, 11))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.4,
            annot_kws={'size': 7})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 Feature vs Target Scatter

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()
for i, feat in enumerate(FEATURES):
    axes[i].scatter(df[feat], df[TARGET], alpha=0.15, s=6, color='#E05C5C')
    axes[i].set_xlabel(feat, fontsize=7)
    axes[i].set_ylabel('Recovery (%)', fontsize=7)
    axes[i].tick_params(labelsize=6)
    # Pearson r
    r = np.corrcoef(df[feat], df[TARGET])[0, 1]
    axes[i].set_title(f'r = {r:.3f}', fontsize=8, fontweight='bold')
for j in range(len(FEATURES), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature vs. Oil Recovery Factor', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig04_feature_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Pre-processing

In [ ]:
X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32).reshape(-1, 1)

# 70 / 15 / 15 split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=VAL_SIZE / (1 - TEST_SIZE), random_state=SEED)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_s = x_scaler.fit_transform(X_train)
X_val_s   = x_scaler.transform(X_val)
X_test_s  = x_scaler.transform(X_test)

y_train_s = y_scaler.fit_transform(y_train)
y_val_s   = y_scaler.transform(y_val)
y_test_s  = y_scaler.transform(y_test)

N_FEATURES = X_train_s.shape[1]
print(f'Input dimension: {N_FEATURES}')

### 3.1 Train / Val / Test Split Visualisation

In [ ]:
sizes  = [X_train.shape[0], X_val.shape[0], X_test.shape[0]]
labels = ['Train (70%)', 'Validation (15%)', 'Test (15%)']
colors = ['#4CAF50', '#FF9800', '#F44336']

fig, ax = plt.subplots(figsize=(7, 4))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts:
    at.set_fontsize(11)
ax.set_title(f'Dataset Split  (n = {len(df)})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig05_dataset_split.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Training Utilities

In [ ]:
def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=10, min_lr=1e-6, verbose=0),
    ]


def fit_model(model, name):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss='mse', metrics=['mae'])
    history = model.fit(
        X_train_s, y_train_s,
        validation_data=(X_val_s, y_val_s),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=get_callbacks(), verbose=0)
    ep  = len(history.history['loss'])
    trL = history.history['loss'][-1]
    vaL = history.history['val_loss'][-1]
    print(f'[{name}] stopped at epoch {ep} — '
          f'train_loss={trL:.5f}  val_loss={vaL:.5f}')
    return history


def evaluate_model(model, name):
    preds_s   = model.predict(X_test_s, verbose=0)
    preds_inv = y_scaler.inverse_transform(preds_s)
    trues_inv = y_scaler.inverse_transform(y_test_s)

    rmse = np.sqrt(mean_squared_error(trues_inv, preds_inv))
    mae  = mean_absolute_error(trues_inv, preds_inv)
    r2   = r2_score(trues_inv, preds_inv)
    mape = np.mean(
        np.abs((trues_inv - preds_inv) /
               np.where(trues_inv == 0, 1e-8, trues_inv))) * 100

    metrics = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}
    print(f'[{name}] RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return preds_inv, trues_inv, metrics

---
## 5. Model 1 — Multi-Layer Perceptron (MLP)

Four fully-connected layers (256 → 128 → 64 → 32) with BatchNorm, ReLU, and Dropout.
The simplest and fastest baseline; often competitive on tabular data.

In [ ]:
def build_mlp(n_features, hidden=(256, 128, 64, 32), dropout=0.3):
    inputs = Input(shape=(n_features,), name='input')
    x = inputs
    for units in hidden:
        x = layers.Dense(units,
                         kernel_regularizer=keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, name='output')(x)
    return Model(inputs, outputs, name='MLP')

mlp_model = build_mlp(N_FEATURES)
mlp_model.summary()

In [ ]:
# Architecture diagram
plot_model(mlp_model, to_file='fig06a_mlp_arch.png',
           show_shapes=True, show_layer_names=True, dpi=96)
from IPython.display import Image
Image('fig06a_mlp_arch.png')

In [ ]:
mlp_history = fit_model(mlp_model, 'MLP')
mlp_preds, mlp_trues, mlp_metrics = evaluate_model(mlp_model, 'MLP')

---
## 6. Model 2 — 1-D Convolutional Neural Network (CNN-1D)

Treats 14 features as a length-14 sequence with 1 channel.
Three Conv1D blocks (32 → 64 → 128 filters) learn local feature interactions;
GlobalAveragePooling compresses the sequence before the regression head.

In [ ]:
def build_cnn(n_features, dropout=0.3):
    inputs = Input(shape=(n_features,), name='input')
    x = layers.Reshape((n_features, 1))(inputs)
    for filters in (32, 64, 128):
        x = layers.Conv1D(filters, kernel_size=3, padding='same',
                          kernel_regularizer=keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(dropout)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, name='output')(x)
    return Model(inputs, outputs, name='CNN_1D')

cnn_model = build_cnn(N_FEATURES)
cnn_model.summary()

In [ ]:
plot_model(cnn_model, to_file='fig06b_cnn_arch.png',
           show_shapes=True, show_layer_names=True, dpi=96)
Image('fig06b_cnn_arch.png')

In [ ]:
cnn_history = fit_model(cnn_model, 'CNN')
cnn_preds, cnn_trues, cnn_metrics = evaluate_model(cnn_model, 'CNN')

---
## 7. Model 3 — LSTM (Long Short-Term Memory)

Each feature is one time-step of a univariate series.
Two stacked LSTM layers with recurrent dropout learn ordered feature interactions
through input, forget, and output gates.

In [ ]:
def build_lstm(n_features, hidden=128, num_layers=2, dropout=0.3):
    inputs = Input(shape=(n_features,), name='input')
    x = layers.Reshape((n_features, 1))(inputs)
    for i in range(num_layers):
        return_seq = (i < num_layers - 1)
        x = layers.LSTM(hidden, return_sequences=return_seq,
                        dropout=dropout)(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, name='output')(x)
    return Model(inputs, outputs, name='LSTM')

lstm_model = build_lstm(N_FEATURES)
lstm_model.summary()

In [ ]:
plot_model(lstm_model, to_file='fig06c_lstm_arch.png',
           show_shapes=True, show_layer_names=True, dpi=96)
Image('fig06c_lstm_arch.png')

In [ ]:
lstm_history = fit_model(lstm_model, 'LSTM')
lstm_preds, lstm_trues, lstm_metrics = evaluate_model(lstm_model, 'LSTM')

---
## 8. Model 4 — Tabular Transformer

Each feature is embedded to *d_model = 64* dimensions.  
A learnable `[CLS]` token prepended to the sequence collects global context  
through 3 encoder blocks of multi-head self-attention (4 heads) + FFN.

In [ ]:
class CLSToken(layers.Layer):
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model

    def build(self, input_shape):
        self.cls = self.add_weight(
            name='cls_token', shape=(1, 1, self.d_model),
            initializer='truncated_normal', trainable=True)
        super().build(input_shape)

    def call(self, x):
        batch = tf.shape(x)[0]
        cls   = tf.tile(self.cls, [batch, 1, 1])
        return tf.concat([cls, x], axis=1)


class TransformerEncoderBlock(layers.Layer):
    def __init__(self, d_model, nhead, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn  = layers.MultiHeadAttention(
            num_heads=nhead, key_dim=d_model // nhead, dropout=dropout)
        self.ffn   = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dropout(dropout),
            layers.Dense(d_model),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn_out = self.attn(x, x, training=training)
        x        = self.norm1(x + self.drop1(attn_out, training=training))
        ffn_out  = self.ffn(x, training=training)
        return self.norm2(x + self.drop2(ffn_out, training=training))


def build_transformer(n_features, d_model=64, nhead=4,
                       num_blocks=3, ff_dim=256, dropout=0.1):
    inputs = Input(shape=(n_features,), name='input')
    x = layers.Reshape((n_features, 1))(inputs)
    x = layers.Dense(d_model, name='feature_embed')(x)
    x = CLSToken(d_model, name='cls_token')(x)

    positions = tf.range(start=0, limit=n_features + 1)
    pos_embed = layers.Embedding(input_dim=n_features + 1,
                                 output_dim=d_model,
                                 name='pos_embed')(positions)
    x = x + pos_embed

    for i in range(num_blocks):
        x = TransformerEncoderBlock(
            d_model, nhead, ff_dim, dropout,
            name=f'enc_block_{i}')(x)

    cls_out = x[:, 0, :]
    cls_out = layers.LayerNormalization()(cls_out)
    cls_out = layers.Dense(64, activation='relu')(cls_out)
    cls_out = layers.Dropout(dropout)(cls_out)
    outputs = layers.Dense(1, name='output')(cls_out)
    return Model(inputs, outputs, name='Transformer')


tfm_model = build_transformer(N_FEATURES)
tfm_model.summary()

In [ ]:
plot_model(tfm_model, to_file='fig06d_tfm_arch.png',
           show_shapes=True, show_layer_names=True, dpi=96)
Image('fig06d_tfm_arch.png')

In [ ]:
tfm_history = fit_model(tfm_model, 'Transformer')
tfm_preds, tfm_trues, tfm_metrics = evaluate_model(tfm_model, 'Transformer')

---
## 9. Comparative Results

In [ ]:
results = pd.DataFrame({
    'Model':    ['MLP', 'CNN-1D', 'LSTM', 'Transformer'],
    'RMSE':     [mlp_metrics['RMSE'],  cnn_metrics['RMSE'],
                 lstm_metrics['RMSE'], tfm_metrics['RMSE']],
    'MAE':      [mlp_metrics['MAE'],   cnn_metrics['MAE'],
                 lstm_metrics['MAE'],  tfm_metrics['MAE']],
    'R²':       [mlp_metrics['R2'],    cnn_metrics['R2'],
                 lstm_metrics['R2'],   tfm_metrics['R2']],
    'MAPE (%)': [mlp_metrics['MAPE'],  cnn_metrics['MAPE'],
                 lstm_metrics['MAPE'], tfm_metrics['MAPE']],
    'Params':   [mlp_model.count_params(), cnn_model.count_params(),
                 lstm_model.count_params(), tfm_model.count_params()],
})

results = results.sort_values('RMSE').reset_index(drop=True)
results.style \
    .background_gradient(subset=['RMSE', 'MAE', 'MAPE (%)'], cmap='RdYlGn_r') \
    .background_gradient(subset=['R²'], cmap='RdYlGn') \
    .format({'RMSE': '{:.4f}', 'MAE': '{:.4f}',
             'R²': '{:.4f}', 'MAPE (%)': '{:.2f}', 'Params': '{:,}'})

### 9.1 Learning Curves

In [ ]:
histories = {'MLP': mlp_history, 'CNN-1D': cnn_history,
             'LSTM': lstm_history, 'Transformer': tfm_history}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in histories.items():
    c = COLOR_MAP[name]
    axes[0].plot(hist.history['loss'],     label=name, color=c, lw=1.8)
    axes[1].plot(hist.history['val_loss'], label=name, color=c, lw=1.8)

for ax, title in zip(axes, ['Training Loss (MSE, scaled)',
                             'Validation Loss (MSE, scaled)']):
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Learning Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig07_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.2 Bar-chart Metric Comparison

In [ ]:
models     = results['Model'].tolist()
bar_colors = [COLOR_MAP[m] for m in models]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (metric, lower) in zip(
        axes, [('RMSE', True), ('MAE', True), ('R²', False), ('MAPE (%)', True)]):
    vals = results[metric].tolist()
    bars = ax.bar(models, vals, color=bar_colors, edgecolor='white', linewidth=1.2)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticklabels(models, rotation=15)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('↓ better' if lower else '↑ better',
                  fontsize=10, color='gray')

plt.suptitle('Test-Set Metric Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig08_metric_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.3 Actual vs. Predicted Scatter

In [ ]:
all_preds = {
    'MLP':         (mlp_trues,  mlp_preds),
    'CNN-1D':      (cnn_trues,  cnn_preds),
    'LSTM':        (lstm_trues, lstm_preds),
    'Transformer': (tfm_trues,  tfm_preds),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, (name, (trues, preds)) in zip(axes.ravel(), all_preds.items()):
    r2   = r2_score(trues, preds)
    rmse = np.sqrt(mean_squared_error(trues, preds))
    ax.scatter(trues, preds, alpha=0.4, s=15,
               color=COLOR_MAP[name], edgecolors='none')
    lo, hi = min(trues.min(), preds.min()), max(trues.max(), preds.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5, label='Perfect fit')
    ax.set_title(f'{name}  (R²={r2:.4f}, RMSE={rmse:.4f})', fontsize=11)
    ax.set_xlabel('Actual Oil Recovery (%)')
    ax.set_ylabel('Predicted Oil Recovery (%)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)

plt.suptitle('Actual vs. Predicted — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig09_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.4 Residual Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, (trues, preds)) in zip(axes.ravel(), all_preds.items()):
    residuals = trues.ravel() - preds.ravel()
    ax.hist(residuals, bins=50, color=COLOR_MAP[name],
            edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'{name} — Residuals  (mean={residuals.mean():.4f})',
                 fontsize=11)
    ax.set_xlabel('Residual (Actual − Predicted)')
    ax.set_ylabel('Count')
    ax.grid(alpha=0.25)

plt.suptitle('Residual Distributions — Test Set',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig10_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.5 Residual vs. Predicted (Homoscedasticity Check)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, (trues, preds)) in zip(axes.ravel(), all_preds.items()):
    residuals = trues.ravel() - preds.ravel()
    ax.scatter(preds.ravel(), residuals, alpha=0.3, s=12,
               color=COLOR_MAP[name], edgecolors='none')
    ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'{name}', fontsize=11)
    ax.set_xlabel('Predicted Oil Recovery (%)')
    ax.set_ylabel('Residual')
    ax.grid(alpha=0.25)

plt.suptitle('Residual vs. Predicted — Homoscedasticity Check',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig11_residual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.6 Absolute Error Distribution (Box Plot)

In [ ]:
abs_errors = {name: np.abs(trues.ravel() - preds.ravel())
              for name, (trues, preds) in all_preds.items()}

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(
    [abs_errors[m] for m in ['MLP', 'CNN-1D', 'LSTM', 'Transformer']],
    labels=['MLP', 'CNN-1D', 'LSTM', 'Transformer'],
    patch_artist=True, notch=False,
    medianprops=dict(color='black', linewidth=2))
for patch, name in zip(bp['boxes'], ['MLP', 'CNN-1D', 'LSTM', 'Transformer']):
    patch.set_facecolor(COLOR_MAP[name])
    patch.set_alpha(0.75)
ax.set_title('Absolute Error Distribution — Test Set',
             fontsize=13, fontweight='bold')
ax.set_ylabel('|Actual − Predicted| (%)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig12_abs_error_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.7 Radar Chart — Multi-metric Overview

In [ ]:
# Normalise each metric to [0, 1] where 1 = best.
# R² is already higher-is-better; RMSE, MAE, MAPE are lower-is-better.
metric_cols   = ['RMSE', 'MAE', 'R²', 'MAPE (%)']
radar_labels  = ['1-RMSE\n(norm)', '1-MAE\n(norm)', 'R²\n(norm)', '1-MAPE\n(norm)']

vals = results[metric_cols].values.astype(float)
norm = np.zeros_like(vals)
for j, col in enumerate(metric_cols):
    col_min, col_max = vals[:, j].min(), vals[:, j].max()
    if col == 'R²':
        norm[:, j] = (vals[:, j] - col_min) / (col_max - col_min + 1e-12)
    else:  # lower is better → invert
        norm[:, j] = 1 - (vals[:, j] - col_min) / (col_max - col_min + 1e-12)

N = len(metric_cols)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for i, name in enumerate(results['Model']):
    vals_r = norm[i].tolist() + norm[i][:1].tolist()
    ax.plot(angles, vals_r, lw=2, color=COLOR_MAP[name], label=name)
    ax.fill(angles, vals_r, alpha=0.08, color=COLOR_MAP[name])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, size=11)
ax.set_ylim(0, 1)
ax.set_title('Radar Chart — Normalised Metrics (outer = better)',
             fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15))
plt.tight_layout()
plt.savefig('fig13_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.8 Parameter Count vs RMSE

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for _, row in results.iterrows():
    name = row['Model']
    ax.scatter(row['Params'], row['RMSE'], s=200,
               color=COLOR_MAP[name], zorder=3, label=name)
    ax.annotate(name, (row['Params'], row['RMSE']),
                textcoords='offset points', xytext=(8, 4), fontsize=10)
ax.set_xlabel('Number of Trainable Parameters', fontsize=11)
ax.set_ylabel('Test RMSE', fontsize=11)
ax.set_title('Model Complexity vs. Accuracy', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig14_complexity_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Feature Importance via Permutation

Shuffle one feature at a time on the test set.  
The RMSE increase measures how much that feature contributes.  
Computed for the best-performing model.

In [ ]:
def permutation_importance(model, X_s, y_inv, y_scaler,
                            feature_names, n_repeats=5):
    base_pred_inv = y_scaler.inverse_transform(
        model.predict(X_s, verbose=0))
    base_rmse = np.sqrt(mean_squared_error(y_inv, base_pred_inv))
    imp = np.zeros((len(feature_names), n_repeats))
    for i in range(len(feature_names)):
        for r in range(n_repeats):
            Xp = X_s.copy()
            np.random.shuffle(Xp[:, i])
            pred_inv  = y_scaler.inverse_transform(
                model.predict(Xp, verbose=0))
            perm_rmse = np.sqrt(mean_squared_error(y_inv, pred_inv))
            imp[i, r] = perm_rmse - base_rmse
    return imp.mean(axis=1), imp.std(axis=1)


best_name  = results.iloc[0]['Model']
model_map  = {'MLP': mlp_model, 'CNN-1D': cnn_model,
              'LSTM': lstm_model, 'Transformer': tfm_model}
best_model = model_map[best_name]
print(f'Best model: {best_name}')

y_test_inv = y_scaler.inverse_transform(y_test_s)
imp_mean, imp_std = permutation_importance(
    best_model, X_test_s, y_test_inv, y_scaler, FEATURES, n_repeats=5)

sorted_idx = np.argsort(imp_mean)[::-1]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Horizontal bar chart
axes[0].barh(
    [FEATURES[i] for i in sorted_idx],
    imp_mean[sorted_idx],
    xerr=imp_std[sorted_idx],
    color=COLOR_MAP[best_name], alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Mean RMSE increase')
axes[0].set_title(f'Permutation Importance — {best_name}',
                  fontsize=12, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Heatmap of importances
imp_df = pd.DataFrame({'Feature': FEATURES,
                        'Importance': imp_mean,
                        'Std': imp_std}).set_index('Feature')
imp_sorted = imp_df.sort_values('Importance', ascending=False)
sns.heatmap(imp_sorted[['Importance']].T, ax=axes[1],
            cmap='YlOrRd', annot=True, fmt='.4f',
            linewidths=0.5, cbar_kws={'label': 'RMSE increase'},
            annot_kws={'size': 7})
axes[1].set_title('Feature Importance Heatmap', fontsize=12, fontweight='bold')
axes[1].set_yticklabels(['Importance'], rotation=0)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right',
                         fontsize=8)

plt.suptitle(f'Feature Importance — {best_name} (Best Model)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig15_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Prediction Error Map (Spatial Distribution of Errors)

In [ ]:
# Sort test samples by actual value to see how errors vary across the range
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (name, (trues, preds)) in zip(axes.ravel(), all_preds.items()):
    idx  = np.argsort(trues.ravel())
    t    = trues.ravel()[idx]
    p    = preds.ravel()[idx]
    err  = np.abs(t - p)
    x    = np.arange(len(t))
    ax.fill_between(x, t, p, alpha=0.35, color=COLOR_MAP[name],
                    label='Error band')
    ax.plot(x, t, 'k-', lw=1.0, label='Actual')
    ax.plot(x, p, '--', lw=1.0, color=COLOR_MAP[name], label='Predicted')
    ax.set_title(f'{name}  —  MAE={err.mean():.4f}', fontsize=11)
    ax.set_xlabel('Sorted test sample index')
    ax.set_ylabel('Oil Recovery (%)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle('Prediction Error Band (sorted by actual value)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig16_error_band.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. Final Summary Table

In [ ]:
print('=' * 70)
print('        FINAL COMPARISON — OIL RECOVERY FACTOR PREDICTION')
print('=' * 70)
print(results[['Model', 'RMSE', 'MAE', 'R²', 'MAPE (%)', 'Params']]
      .to_string(index=False))
print()
print('Best model by RMSE :', results.iloc[0]['Model'])
print('Best model by R²   :', results.loc[results['R²'].idxmax(), 'Model'])
print('Most compact model :', results.loc[results['Params'].idxmin(), 'Model'])

---
## Discussion

| Model | Strength | Weakness |
|---|---|---|
| **MLP** | Fast, simple, strong baseline on tabular data | No local/sequential feature correlations |
| **CNN-1D** | Efficient local feature interactions via sliding filters | Feature order arbitrary; no long-range dependencies |
| **LSTM** | Gated memory captures ordered feature interactions | Slower training; less parallelisable |
| **Transformer** | Full pairwise self-attention; most expressive | Higher memory; needs careful tuning |

### Figures produced
| File | Content |
|---|---|
| `fig01_target_distribution.png` | Histogram, box plot, CDF of target |
| `fig02_feature_distributions.png` | Histogram for each of the 14 features |
| `fig03_correlation_heatmap.png` | Lower-triangular correlation matrix |
| `fig04_feature_vs_target.png` | Scatter + Pearson r for each feature vs target |
| `fig05_dataset_split.png` | Train / val / test pie chart |
| `fig06a–d_*_arch.png` | Keras architecture diagrams for each model |
| `fig07_learning_curves.png` | Train & val MSE curves for all 4 models |
| `fig08_metric_bars.png` | Side-by-side bar charts (RMSE, MAE, R², MAPE) |
| `fig09_actual_vs_predicted.png` | Scatter: actual vs predicted (test set) |
| `fig10_residuals.png` | Residual histograms |
| `fig11_residual_vs_predicted.png` | Residual vs predicted (homoscedasticity) |
| `fig12_abs_error_boxplot.png` | Absolute error box plots |
| `fig13_radar_chart.png` | Radar / spider chart of normalised metrics |
| `fig14_complexity_vs_accuracy.png` | Parameter count vs RMSE scatter |
| `fig15_feature_importance.png` | Permutation importance bar + heatmap |
| `fig16_error_band.png` | Prediction error bands sorted by actual value |